<a href="https://colab.research.google.com/github/mattsanchez282/CPSC_475_notebooks/blob/main/Copy_of_nonlinear_regression_and_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nonlinear Regression + kNN Classwork

If you are on Google Colab, run this cell first to install Lets-Plot and the UCI ML Repository

In [ ]:
!pip install lets-plot
!pip install ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 5.1 MB/s eta 0:00:00


All the imports for this notebook

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from lets_plot import *

We will be using the Auto MPG dataset (again), which gives the attributes of a set of cars.

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
auto = fetch_ucirepo(id=9).data.original

# A quirk of the dataset is that "origin" should be regions
# But the data is stored as 1, 2, and 3
# I'll fix that for you here
auto["origin"] = auto["origin"].map({
    1: "USA",
    2: "Europe",
    3: "Japan"
})

# Impute the missing horsepower values with the median
auto["horsepower"] = auto["horsepower"].fillna(auto["horsepower"].median())

# The car names are the identifiers of each instance
auto = auto.set_index('car_name')
auto.head()

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,mpg
car_name,,,,,,,,
"chevrolet,chevelle,malibu",8,307.0,130.0,3504,12.0,70,USA,18.0
"buick,skylark,320",8,350.0,165.0,3693,11.5,70,USA,15.0
"plymouth,satellite",8,318.0,150.0,3436,11.0,70,USA,18.0
"amc,rebel,sst",8,304.0,150.0,3433,12.0,70,USA,16.0
"ford,torino",8,302.0,140.0,3449,10.5,70,USA,17.0


I will also one-hot encode the "origin" variable for you and store it into the `auto_encoded` DataFrame. (We did this last time)

In [ ]:
auto_encoded = pd.get_dummies(auto, columns=["origin"], drop_first=True, dtype=int)
auto_encoded.head()

,cylinders,displacement,horsepower,weight,acceleration,model_year,mpg,origin_Japan,origin_USA
car_name,,,,,,,,,
"chevrolet,chevelle,malibu",8,307.0,130.0,3504,12.0,70,18.0,0,1
"buick,skylark,320",8,350.0,165.0,3693,11.5,70,15.0,0,1
"plymouth,satellite",8,318.0,150.0,3436,11.0,70,18.0,0,1
"amc,rebel,sst",8,304.0,150.0,3433,12.0,70,16.0,0,1
"ford,torino",8,302.0,140.0,3449,10.5,70,17.0,0,1


# Part 1 - Nonlinear Transformations

Create the transformation
$$\frac{1}{displacement^2}$$
as a new column in `auto_encoded`.

In [ ]:
auto_encoded['transformation'] = 1 / (auto_encoded["displacement"] ** 2)

Create a multiple regression model of the form:

$$mpg = \theta_0 + \theta_1(displacement) + \theta_2\left(\frac{1}{displacement^2}\right)$$



Plot the learned nonlinear function over the scatter plot of displacement vs mpg.

# Part 2 - k-Nearest Neighbors

The question we will answer with kNN: **Can we predict whether a car was made in the USA, Europe, or Japan based on its physical characteristics?**

To start this exercise, I will scale the data, and split the data for you into a training and testing set. I'll also split out the validation set for you. Since I'm doing it for you this time, just pay attention to the syntax here!


In [11]:
random_seed = 42 # Change it to whatever you want!

from sklearn.model_selection import train_test_split

# The data we are interested in
X = auto[["mpg", "horsepower", "weight", "displacement"]]
y = auto["origin"]

# Use 0-1 scaling
X = (X - X.min()) / (X.max() - X.min())

# First, separate out a test set
X, X_test, y, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_seed,
    stratify=y
)

# Then, split out a validation set from the training set X
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_seed,
    stratify=y
)

The above code gives you:
*   `X` and `y` - Your training data for your final kNN
*   `X_test` and `y_test` - Your test data for your final kNN
*   `X_train` and `y_train` - Your training data when you are trying to find the right $k$
*   `X_val` and `y_val` - Your validation data when you are trying to find the right $k$



For the code below, fill in everything that says `pass`, per the instructions in the comments above them. The below code cell will create a plot of $k$ vs validation error (if you implemented everything correctly)

In [15]:
from sklearn.neighbors import KNeighborsClassifier

# Put down a list of values of k you want to test out in this list
ks = range(1,15)

scores = np.array([])

for k in ks:
    # Using X_train and y_train, fit a kNN classifier for each k, and store the
    # accuracy from X_val and y_val into a "score" variable
    knn = KNeighborsClassifier(n_neighbors=k, metric = 'manhattan')
    knn.fit(X_train, y_train)
    score = knn.score(X_val, y_val)

    scores = np.append(scores, score) # Adds the validation score

# Plot your validation performance over k
scores_df = pd.DataFrame({"k": ks, "error": 1 - scores})
ggplot(scores_df)\
    + geom_line(mapping=aes(x="k", y="error"))

Determine what value of $k$ you want to use using the elbow method. Once you do, train your final kNN classifier using `X` and `y`. Test out how accurate your kNN is by printing out its accuracy on `X_test` and `y_test`!

In [16]:
knn_final = KNeighborsClassifier(n_neighbors=3, metric = 'manhattan')
knn_final.fit(X, y)
knn_final.score(X_test, y_test)

0.8125

*OPTIONAL TASK*: If you have time, create a scatter plot of "weight" vs "mpg" in the **test set**, color-coding each point by their predictions, and having the shapes of each point correspond to whether the kNN model predicted the point correctly or not.

You are done! Submit this for attendance on Canvas 🎉